# noise_reachability step 6: do the labels and arm B's uneven gain hold beyond task 0? (Kaggle, 2 x T4)

Step 6 of `noise_reachability/README.md` (design fixed 2026-10-31). The base `lerobot/smolvla_libero` at `a24f1f3804`
and step 4's arm B checkpoint, `n_action_steps=10`, on tasks 4, 6, 7, 8 x init states 0-9 x seeds 16-31: 2 policies x
640 episodes. No training.

**Before the first version:** add the continued-training checkpoint as input (**Add Input > Your Datasets**), as in step
4. Step 7 finds it by the job name in its `train_config.json` and checks it is step 4's.

**Run with Save & Run All (Accelerator: GPU T4 x2, Internet on).** Nothing needs editing. The base runs first. Step 9
stops the run if a task's base differs from seeds 0-15 by more than 18 of 160. Then B, then the analysis. If anything
stops the run, step 12 still writes the zip, then reports the failure as an error.

**If step 10 says work remains**, add this version's output as input (**Add Input > Your Work**, keep the checkpoint
dataset) and Save & Run All again: finished episodes are skipped.

Download from the Output tab: `step6_results.zip`.

## 1. Settings

In [ ]:
import csv, glob, hashlib, json, os, re, shutil, signal, subprocess, sys, time
from pathlib import Path

T_SESSION = time.time()
REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "24437d2d821384f2447b85d7b863f9b793a23c0a"
assert REPO_REF != "SET_AFTER_COMMIT", "this notebook has no reviewed commit yet"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

# The protocol of README.md step 6; nothing here is meant to be edited.
CHECKPOINT_REPO, CHECKPOINT_REVISION = "lerobot/smolvla_libero", "a24f1f380461b665158d52acd140152c79e8b36d"
B_JOB_NAME = "smolvla_libero10_continued"     # the dataset-input folder whose train_config.json names this job
B_SHA256 = "b75872e158799eb4c824d2ac8d1f6e956bb0cc243d9033d302fa8e1868d6b336"  # step 4's arm_b_checkpoint.json
TASKS = [4, 6, 7, 8]
N_ACTION_STEPS = 10
EVAL_SEED_PARTS = ["16-19", "20-23", "24-27", "28-31"]  # 4 x 4 seeds = 16 per init state
EVAL_INIT_STATES = "0-9"
PART_EPISODES = 40
OUT_NAME = "step6"
OUT_ROOT = f"/kaggle/working/{OUT_NAME}"
STOP_AT_H = 10.5                   # no new work after this many hours of the session; running jobs are stopped
GB_PER_JOB = 6                     # base policy + one episode in RAM per simulator process

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"
print(f"step 6 at {REPO_REF}: tasks {TASKS}, n_action_steps {N_ACTION_STEPS}, policies base and B, seeds 16-31 x init "
      f"states {EVAL_INIT_STATES}, {2 * len(TASKS) * len(EVAL_SEED_PARTS)} jobs, {2 * len(TASKS) * 160} episodes, output {OUT_ROOT}")


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r


def hours_left():
    return STOP_AT_H - (time.time() - T_SESSION) / 3600

## 2. Hardware

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus, "No GPU. Set Accelerator to GPU T4 x2."
for name, mem, cap, driver in gpus:
    assert float(cap) >= 7.5, f"{name} (compute capability {cap}) is not supported by the pinned PyTorch build; use GPU T4 x2"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
mem_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(2 * len(gpus), int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
GPU_IDS = ",".join(str(i) for i in range(len(gpus)))
print(f"{len(gpus)} GPU(s), driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"RAM {mem_gb:.0f} GB ({avail_gb:.0f} GB available) -> {JOBS} evaluations at a time on GPUs {GPU_IDS}")
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"
assert JOBS >= 4, f"RAM for {JOBS} SmolVLA processes; the notebook runs 4 at once"

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

The package versions of the local `lerobot_env` (as the earlier Kaggle notebooks). Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks, tests and the base checkpoint

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", OMP_NUM_THREADS="1", PYTHONPATH=REPO,
               PYTHONFAULTHANDLER="1")
sh(f"""{PY} -c "
import torch, av
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], '| av', av.__version__)
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)
r = sh(f"""{PY} -c "from huggingface_hub import snapshot_download; print(snapshot_download('{CHECKPOINT_REPO}', revision='{CHECKPOINT_REVISION}'))" """, env=RUN_ENV)
POLICY_PATH = r.stdout.strip().splitlines()[-1]
MAIN = f"{REPO}/self_filtered_bc/self_filtered_bc.py"
print("base checkpoint", POLICY_PATH)

## 6. Resume from an earlier version's output

In [ ]:
os.makedirs(OUT_ROOT, exist_ok=True)
for prev in sorted(glob.glob(f"/kaggle/input/**/{OUT_NAME}", recursive=True)):
    if os.path.isdir(prev) and os.path.exists(f"{prev}/repo_commit.txt"):
        print("copying", prev)
        shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
head = sh(f"git -C {REPO} log -1 --format='%H'").stdout.strip()
first = Path(OUT_ROOT, "repo_commit.txt")
started_with = first.read_text().split()[0] if first.exists() and first.read_text().strip() else None
if started_with and started_with != head:
    raise RuntimeError(f"the run started at commit {started_with}, this version cloned {head}: set REPO_REF in "
                       f"step 1 to {started_with}")
with open(first, "a") as f:
    f.write(head + "\n")
for d in ("logs", "eval"):
    os.makedirs(f"{OUT_ROOT}/{d}", exist_ok=True)

## 7. Arm B's checkpoint (Kaggle dataset input)

In [ ]:
# As baseline_performance/kaggle_baseline_lerobot_eval.ipynb: found by its training job, whatever the folder is called.
found = [str(Path(f).parent) for f in glob.glob("/kaggle/input/**/train_config.json", recursive=True)
         if json.load(open(f)).get("job_name") == B_JOB_NAME and Path(f).with_name("model.safetensors").exists()]
assert len(found) == 1, (f"need exactly one input folder whose train_config.json has job_name {B_JOB_NAME} and a "
                         f"model.safetensors, found {found}: add the continued-training checkpoint as a dataset input")
B_PATH = found[0]
h = hashlib.sha256()
with open(f"{B_PATH}/model.safetensors", "rb") as f:
    for block in iter(lambda: f.read(1 << 24), b""):
        h.update(block)
B_INFO = {"path": B_PATH, "model_sha256": h.hexdigest(), "files": sorted(os.listdir(B_PATH))}
prev = Path(OUT_ROOT, "arm_b_checkpoint.json")
if prev.exists():
    assert json.loads(prev.read_text())["model_sha256"] == B_INFO["model_sha256"], "arm B's checkpoint changed between versions"
prev.write_text(json.dumps(B_INFO, indent=1) + "\n")
print(json.dumps(B_INFO, indent=1))
assert B_INFO["model_sha256"] == B_SHA256, "this is not the checkpoint step 4 evaluated"

## 8. Helpers: parallel jobs with resume and a deadline

In [ ]:
def run_jobs(jobs, max_parallel, poll_s=30, log_every_s=600):
    """Run {name: (command, gpu)} with at most max_parallel at once; exit code 3 means 'call again' (more
    episodes in a fresh process). No job starts after the deadline and running ones are stopped at it.
    Returns {name: exit code}; a job stopped at the deadline gets None."""
    pending, running, codes, last_log = list(jobs), {}, {}, time.time()
    while pending or running:
        while pending and len(running) < max_parallel and hours_left() > 0:
            name = pending.pop(0)
            cmd, gpu = jobs[name]
            log = open(f"{OUT_ROOT}/logs/{name}.log", "a")
            env = dict(RUN_ENV, CUDA_VISIBLE_DEVICES=str(gpu))
            running[name] = (subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=env, stdout=log,
                                              stderr=subprocess.STDOUT, start_new_session=True), log)
            time.sleep(5)  # staggered EGL start-up (FAIL-006)
        if hours_left() <= 0 and running:
            print(time.strftime("%H:%M"), "deadline: stopping", list(running), flush=True)
            for name, (proc, log) in running.items():
                if proc.poll() is None:
                    os.killpg(proc.pid, signal.SIGTERM)
            time.sleep(20)
            for name, (proc, log) in running.items():
                log.close()
                codes[name] = None
            for name in pending:
                codes[name] = None
            return codes
        if hours_left() <= 0 and not running:
            for name in pending:
                codes[name] = None
            return codes
        for name, (proc, log) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[name]
            if proc.returncode == 3:
                pending.insert(0, name)
                continue
            codes[name] = proc.returncode
            print(time.strftime("%H:%M"), name, "exit", proc.returncode, flush=True)
        if time.time() - last_log > log_every_s:
            last_log = time.time()
            print(time.strftime("%H:%M"), f"{hours_left():.1f} h left; running {list(running)}; "
                  f"{len(pending)} waiting", flush=True)
        time.sleep(poll_s)
    return codes


FIELDS = ["task_id", "seed", "init_state", "success", "steps"]


def read_csv(path):
    return list(csv.DictReader(open(path))) if os.path.exists(path) else []


def write_csv(rows, out):
    os.makedirs(os.path.dirname(out), exist_ok=True)
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows({k: r[k] for k in FIELDS} for r in rows)
    return rows


def n_range(spec):
    a, b = (spec.split("-") + [spec])[:2]
    return int(b) - int(a) + 1


def failed(codes):
    return {k: c for k, c in codes.items() if c not in (0, None)}


FAILURE = None


def fail(message):
    """Keep going to the zip in step 12, which raises this."""
    global FAILURE
    FAILURE = message
    print("FAILED:", message, flush=True)


LABEL_FILES = (f"{REPO}/week6/noise_reachability_step0_kaggle/nas10_all_tasks/per_episode.csv "
               f"{REPO}/week6/noise_reachability_step1_kaggle/nas10_step1/per_episode.csv")
ANALYZE = f"cd {REPO} && {PY} -S noise_reachability/analyze_step6.py --labels {LABEL_FILES}"


def part_dir(policy, task, j):
    return f"{OUT_ROOT}/eval/{policy}/task_{task}/part{j}"


def rows_of(policy):
    return [r for t in TASKS for j in range(len(EVAL_SEED_PARTS))
            for r in read_csv(f"{part_dir(policy, t, j)}/per_episode_base.csv")]


def evaluate(policy, path):
    """Both policies load with --eval_policies base, as step 4 ran arm B."""
    jobs = {}
    for t in TASKS:
        for j, part in enumerate(EVAL_SEED_PARTS):
            out = part_dir(policy, t, j)
            if len(read_csv(f"{out}/per_episode_base.csv")) < PART_EPISODES:
                jobs[f"eval_{policy}_task{t}_part{j}"] = (
                    f"cd {REPO} && {PY} -u {MAIN} --stage eval --task_id {t} --n_action_steps {N_ACTION_STEPS} "
                    f"--policy_path {path} --eval_policies base --eval_seeds {part} "
                    f"--eval_init_states {EVAL_INIT_STATES} --eval_output_dir {out} --max_episodes_per_process 10",
                    len(jobs) % 2)
    print(f"{policy}: {len(jobs)} jobs to run; {hours_left():.1f} h left")
    if jobs and hours_left() > 0:
        codes = run_jobs(jobs, max_parallel=4)
        if failed(codes):
            fail(f"evaluation failed: {failed(codes)}")
            return False
    rows = rows_of(policy)
    print(f"{policy}: {len(rows)}/{160 * len(TASKS)} episodes")
    if len(rows) == 160 * len(TASKS):
        write_csv(rows, f"{OUT_ROOT}/eval/per_episode_{policy}.csv")
        return True
    return False

## 9. The base, then the harness

Stops the run if a task's base differs from its seeds 0-15 count by more than 18 of 160 (README, step 6).

In [ ]:
BASE_DONE = evaluate("base", POLICY_PATH)
if BASE_DONE and sh(f"{ANALYZE} --base {OUT_ROOT}/eval/per_episode_base.csv --harness_only", check=False,
                    env=RUN_ENV).returncode != 0:
    fail("the harness failed: investigate before running B")
    BASE_DONE = False

## 10. Arm B

In [ ]:
B_DONE = evaluate("B", B_PATH) if BASE_DONE else False
if not FAILURE:
    print("all episodes done" if B_DONE else "WORK REMAINS: add this version's output as input and Save & Run All again")

## 11. Analysis (when every episode is done)

In [ ]:
if B_DONE:
    STEP4 = f"{REPO}/week6/noise_reachability_step4_kaggle/run/eval"
    if sh(f"{ANALYZE} --base {OUT_ROOT}/eval/per_episode_base.csv --b {OUT_ROOT}/eval/per_episode_B.csv "
          f"--task0_base {STEP4}/per_episode_base.csv --task0_b {STEP4}/per_episode_B.csv "
          f"--out {OUT_ROOT}/step6_report.txt", check=False, env=RUN_ENV).returncode != 0:
        fail("the analysis failed: the episodes are in the zip")
else:
    print("evaluation not finished; analysis skipped")

## 12. Files for the repo

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /kaggle/working/step6_results.zip . -x '*.npz' '*.safetensors' '*.pt'")
sh(f"du -h /kaggle/working/step6_results.zip; du -sh {OUT_ROOT}")
print(f"session time used: {(time.time() - T_SESSION) / 3600:.1f} h")
if FAILURE:
    raise RuntimeError(FAILURE)